# 🧱 Data Analyst in Databricks — Portfolio Showcase

![Databricks](https://img.shields.io/badge/Platform-Databricks-FF3621?logo=databricks&logoColor=white)
![SQL](https://img.shields.io/badge/Language-SQL-4479A1?logo=postgresql&logoColor=white)
![Level](https://img.shields.io/badge/Level-Intermediate-yellow)
![Status](https://img.shields.io/badge/Status-Completed-brightgreen)

> **Course:** Data Analyst in Databricks  
> **Platform:** [DataCamp](https://www.datacamp.com/)

This notebook is a curated showcase of the hands-on work completed across the four modules of this track: data engineering fundamentals, Databricks SQL, Delta Lake data management, and data visualization/dashboarding. For every module it summarizes the key concepts, highlights representative SQL exercises solved on the platform, and includes screenshots of the actual results produced in the Databricks workspace.

Full notes, official course PDFs, and complete exercise screenshots are kept in each module's folder — this notebook is the highlight reel.

---


## 🗂️ Module Structure

| # | Folder | Module | Key Topics |
|---|--------|--------|-------------|
| 1 | [`1_Understanding Data Engineering`](./1_Understanding%20Data%20Engineering) | Understanding Data Engineering | Data engineering vs. data science, data pipelines, structured/unstructured data, data warehouses & lakes, the 5 Vs of big data |
| 2 | [`7_Introduction to Databricks SQL`](./7_Introduction%20to%20Databricks%20SQL) | Introduction to Databricks SQL | SQL Editor, Unity Catalog, queries, visualizations, ingesting & cleaning data (Bronze → Silver → Gold), dashboards, Partner Connect |
| 3 | [`8_Data Management in Databricks`](./8_Data%20Management%20in%20Databricks) | Data Management in Databricks | Delta Lake & ACID transactions, managed vs. unmanaged tables, views & temp views, data exploration, access control & PII |
| 4 | [`9_Data Visualization in Databricks`](./9_Data%20Visualization%20in%20Databricks) | Data Visualization in Databricks | Chart types, formatting & data storytelling, dashboards, filters/parameters, alerts & sharing |

Each folder contains the official chapter PDFs (`chapterN.pdf`), personal notes (`ChapterN-*.txt`), and a Word document with the full set of exercise screenshots (`*Screenshots.docx`).

---


## 📌 Module 1 — Understanding Data Engineering
> Folder: [`1_Understanding Data Engineering`](./1_Understanding%20Data%20Engineering)

A conceptual, no-coding module laying the theoretical foundation for the rest of the track.

**Key concepts**
- **Data engineers vs. data scientists:** data engineers ingest data from different sources, optimize databases for analysis, remove corrupted data, and build/maintain data architectures so that data scientists and analysts can focus on extracting insights.
- **Data pipelines:** the flow of data from raw sources to analysis-ready outputs.
- **Structured vs. unstructured data:** structured data is consistent, typed, and stored in relational databases queried with SQL (~20% of all data); unstructured data lacks a fixed schema.
- **Data warehouses vs. data lakes:** warehouses are optimized for structured, SQL-based analytics; lakes store raw data of any type at lower cost with more flexibility.
- **The 5 Vs of big data:** Volume, Variety, Velocity, Veracity, and Value — the framework used to reason about why traditional methods break down at scale.

This module set up the vocabulary and mental model used throughout the rest of the specialization, particularly the **Bronze → Silver → Gold (medallion architecture)** pattern applied hands-on in Module 2.

---


## 📌 Module 2 — Introduction to Databricks SQL
> Folder: [`7_Introduction to Databricks SQL`](./7_Introduction%20to%20Databricks%20SQL)

Hands-on introduction to querying, transforming, and visualizing data directly inside the Databricks Lakehouse using an **insurance claims dataset**.

**Key concepts**
- The Lakehouse combines the low cost/flexibility of data lakes with the performance and SQL support of data warehouses (Delta format, ANSI SQL, BI-tool integrations via Partner Connect).
- **Databricks SQL key assets:** Queries (the base unit of analysis), SQL Warehouses (dedicated SQL compute), Tables vs. Views, and Visualizations vs. Dashboards.
- **Medallion architecture in practice:** cleaning raw data into the **Silver** layer (removing NULLs, standardizing values, fixing types) and aggregating it into the **Gold** layer (KPIs, business-ready views).

**Exercise highlights**
- Filtered the `insurance` table for rented houses with high risk segmentation → **505 matching claims**.
- Built the `Average_Premiums` query and visualization → identified **OK (lowest)** and **GA (highest)** average premium states; Arizona's average premium came out to **$89.47**.
- Cleaned a messy `employee_data` table with `CASE` + `CURRENT_DATE()` logic and promoted it to a Silver table (`employee_data_silver`).
- Joined insurance and employee data into `agent_customer_silver`, casting account/routing numbers to `STRING`.
- Built the **large_claims** table (claims ≥ $100) — **1,263 Property-insurance claims** met the criteria.
- Built the **Agent Analysis Dashboard**, joining insurance + employee data, adding a bar chart, histogram, date-range filter, and a `LIKE`-based `agentName` parameter → surfaced that agents named "Steve" had their highest claim amounts in **Property** insurance.
- Built a **Leadership KPI Dashboard** with counter widgets for total claims, total claim amount, and average premium, filterable by insurance type.

```sql
-- Cleaning raw data into the Silver layer
CREATE TABLE employee_data_silver AS (
    SELECT DISTINCT *,
        CASE WHEN DATE_OF_JOINING IS NOT NULL THEN DATE_OF_JOINING
             ELSE CURRENT_DATE()
        END AS JOINING_DATE
    FROM employee_data_messy
);

-- Aggregating into a Gold-ready large-claims table
CREATE TABLE large_claims AS
SELECT COUNT(DISTINCT CUSTOMER_ID) AS customer_count,
       INSURANCE_TYPE
FROM insurance
WHERE PREMIUM_AMOUNT >= 100   -- proxy for a "large claim"
GROUP BY INSURANCE_TYPE;
```

**SQL Editor — filtering the insurance table**

![SQL Editor filtering insurance claims](./assets/sql_editor_insurance_filter.png)

**Agent Analysis Dashboard**

![Agent Analysis Dashboard](./assets/agent_analysis_dashboard.png)

---


## 📌 Module 3 — Data Management in Databricks
> Folder: [`8_Data Management in Databricks`](./8_Data%20Management%20in%20Databricks)

Focused on **Delta Lake** fundamentals using a healthcare dataset (patients, prescriptions, appointments).

**Key concepts**
- **ACID transactions** (Atomicity, Consistency, Isolation, Durability) guarantee reliable, consistent updates — critical in a healthcare context where incomplete records or race conditions are unacceptable.
- **Schema enforcement & evolution**, plus Delta Lake's **time travel** feature to access previous versions of a table for historical/audit review.
- **Managed vs. unmanaged tables:** managed tables have their lifecycle (including underlying data) fully controlled by Databricks; unmanaged tables use a custom `LOCATION`, so dropping the table doesn't delete the underlying files — useful for compliance and external storage (S3, Azure Blob).
- **Views vs. temp views:** views persist and are reusable across sessions (great for dashboards/reports); temp views exist only for the current session (great for staging/exploration).
- **Data Explorer & governance:** navigating catalogs/schemas/tables, managing table ownership, controlling read/write permissions, and handling **PII** in line with regulations like GDPR and HIPAA.

**Exercise highlights**
- Updated a patient's phone number directly in a Delta table and verified the change with `SELECT *`, confirming ACID-safe updates.
- Queried and then **deleted** a specific prescription record (`prescription_id = 1001`, medication `'approach'`) from the `prescriptions` Delta table.
- Ran `OPTIMIZE` on the `appointments` table to compact small Parquet files and compared query performance/file counts before and after.
- Created a **managed table** (`patients_managed`) from an existing Delta table to compare storage/lifecycle behavior against unmanaged tables.
- Practiced `CREATE VIEW` / `CREATE OR REPLACE TEMP VIEW` to build reusable and session-scoped views (e.g. `active_patients_view`, `patient_blood_group_view`).

```sql
-- ACID-safe update
UPDATE patients
SET phone_number = '+1-555-123-4567'
WHERE patient_id = 3;

-- Compacting small files for better read performance
OPTIMIZE appointments;

-- Managed table created from an existing Delta table
CREATE TABLE patients_managed AS
SELECT * FROM patients;
```

**Deleting a record from the `prescriptions` Delta table**

![Deleting a record from a Delta table](./assets/delta_lake_delete_record.png)

**Checking views created during the module (`SHOW VIEWS`)**

![Views created in the workspace](./assets/views_show_views.png)

---


## 📌 Module 4 — Data Visualization in Databricks
> Folder: [`9_Data Visualization in Databricks`](./9_Data%20Visualization%20in%20Databricks)

Applied chart design, dashboarding, and data-storytelling principles using the classic **NYC taxi trips** dataset and a retail sales dataset.

**Key concepts**
- Discrete vs. continuous data, and how descriptive statistics (mean, median, distributions) inform the right visualization choice.
- **Chart types and when to use them:** bar (compare categories), line (trends over time), pie (part-to-whole), scatter (correlations), heat maps (intensity patterns), histograms (distributions), choropleth/marker maps (geography).
- **Formatting best practices:** consistent bar types and simple color schemes over "rainbow" palettes or 3D effects, clear axes, and precise data labels — functionality over aesthetics.
- **Data storytelling:** turning a cluttered, raw chart into a clear narrative that highlights the specific insight the audience needs.
- **Dashboards:** combining multiple visualizations, filters, and parameters into a single, shareable, auto-refreshing view; managing dashboards via cloning, exporting, sharing (View/Edit/Manage permissions), and setting up alerts.

**Exercise highlights**
- Built a **bar chart** of average fare by pickup hour → **5 AM** had the highest average fare.
- Built a **line chart** of fare standard deviation by hour → **7 PM** had the most stable (lowest std-dev) pricing.
- Built a **combo chart** (trip distance + fare amount by hour) to compare volume and revenue trends simultaneously.
- Built a **choropleth map** of retail sales by country → the **United Kingdom** had the highest total sales.
- Assembled an interactive NYC Taxi dashboard with counter widgets, scatter plots, and pickup-zip / day-of-week filters.

```sql
-- Base query used across the taxi-fare visualizations
SELECT *,
       hour(tpep_pickup_datetime)  AS pickup_hour,
       month(tpep_pickup_datetime) AS month
FROM samples.nyctaxi.trips;
```

**Combo chart — fare amount & trip distance by hour**

![Combo chart of NYC taxi fares](./assets/combo_chart_taxi.png)

**Dashboard with counter widget (total trips)**

![Dashboard counter widget](./assets/dashboard_counter_widget.png)

**Interactive dashboard with filters (pickup zip, day of week)**

![NYC taxi dashboard with filters](./assets/dashboard_filters_nyc_taxi.png)

---


## ✅ Takeaways

Across this track I went from the conceptual foundations of data engineering to hands-on, end-to-end work inside the Databricks Lakehouse: writing ANSI SQL against Unity Catalog, building Bronze → Silver → Gold pipelines, managing Delta Lake tables safely with ACID guarantees, and turning query results into dashboards that answer real business questions (risk segmentation, churn risk, agent performance, regional sales).

*Built with 🤍 as part of a continuous learning journey in Data Analytics.*
